In [2]:
import torch
import pandas as pd
from torch_geometric.data import HeteroData
import torch_geometric.transforms as T
from torch_geometric.loader import LinkNeighborLoader
from torch_geometric.nn import SAGEConv, to_hetero


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [20]:
def build_type_maps(df):
    maps = {}
    for t in pd.concat([df["type_entity_1"], df["type_entity_2"]]).unique():
        ids = pd.concat([
            df.loc[df["type_entity_1"] == t, "id_entity_1"],
            df.loc[df["type_entity_2"] == t, "id_entity_2"],
        ]).unique()
        maps[t] = {int(rid): i for i, rid in enumerate(ids)}
    return maps

def init_nodes(data, maps, emb_dim=64, device="cpu"):
    for node_type, m in maps.items():
        n = len(m)
        data[node_type].x = torch.randn(n, emb_dim, device=device)
    return data


def add_edges(data, df, maps, device="cpu"):
    # сгруппируем по (type1, pred, type2)
    for (t1, pred, t2), g in df.groupby(["type_entity_1", "predicate", "type_entity_2"], sort=False):
        src = torch.tensor([maps[t1][int(x)] for x in g["id_entity_1"].tolist()],
                           dtype=torch.long, device=device)
        dst = torch.tensor([maps[t2][int(x)] for x in g["id_entity_2"].tolist()],
                           dtype=torch.long, device=device)
        ei = torch.stack([src, dst], dim=0)  # [2, E]
        data[(t1, pred, t2)].edge_index = ei

    return data


def df_to_heterodata(df, emb_dim=64, device="cpu"):
    df = df.copy()
    maps = build_type_maps(df)

    data = HeteroData()
    data = init_nodes(data, maps, emb_dim=emb_dim, device=device)
    data = add_edges(data, df, maps, device=device)

    return data, maps


In [21]:
df = pd.read_csv('../data/edges/clean_triples.csv')

In [22]:
data, maps = df_to_heterodata(df, emb_dim=16, device="cuda")
data = T.ToUndirected()(data)

In [23]:
target_edges = [edge for edge in data.edge_types if edge[1] == 'interacts_with']
rev_target_edges = [edge for edge in data.edge_types if edge[1] == 'rev_interacts_with']

transform = T.RandomLinkSplit(
    num_val=0.1,
    num_test=0.1,
    neg_sampling_ratio=1.0,
    is_undirected = True,
    # Указываем только название типа связи (строку)
    edge_types= ('AA', 'interacts_with', 'SmallMolecule'), 
    rev_edge_types= ('SmallMolecule', 'rev_interacts_with', 'AA'), # Если есть обратные связи
)

In [24]:
train_data, val_data, test_data = transform(data)

In [25]:
edge_label_index = train_data['AA', 'interacts_with', 'SmallMolecule'].edge_label_index
edge_label = train_data['AA', 'interacts_with', 'SmallMolecule'].edge_label

train_loader = LinkNeighborLoader(
    data=train_data,
    num_neighbors=[15, 10],
    edge_label_index=(('AA', 'interacts_with', 'SmallMolecule'), edge_label_index),
    edge_label=edge_label,
    batch_size=128,
    shuffle=True,
    neg_sampling_ratio=1.0, # Добавляет столько же "фейковых" связей для обучения
)

/home/ivanc/projects/gnn/.venv/lib/python3.11/site-packages/torch_geometric/loader/link_neighbor_loader.py:252: UserWarning: Using 'NeighborSampler' without a 'pyg-lib' installation is deprecated and will be removed soon. Please install 'pyg-lib' for accelerated neighborhood sampling
  neighbor_sampler = NeighborSampler(


In [17]:
class GNN(torch.nn.Module):
    def __init__(self, hidden_channels):
        super().__init__()
        self.conv1 = SAGEConv((-1, -1), hidden_channels)
        self.conv2 = SAGEConv((-1, -1), hidden_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index)
        return x

# Превращаем модель в гетерогенную на основе метаданных вашего графа
model = GNN(hidden_channels=32)
model = to_hetero(model, data.metadata(), aggr='mean')

class Classifier(torch.nn.Module):
    def forward(self, x_head, x_tail, edge_label_index):
        # Собираем эмбеддинги для пар узлов, между которыми предсказываем связь
        edge_feat_head = x_head[edge_label_index[0]]
        edge_feat_tail = x_tail[edge_label_index[1]]
        # Считаем схожесть
        return (edge_feat_head * edge_feat_tail).sum(dim=-1)
    
classifier = Classifier()

In [18]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = torch.nn.BCEWithLogitsLoss()

def train():
    model.train()
    for batch in train_loader:
        optimizer.zero_grad()
        
        # 1. Получаем эмбеддинги для всех типов узлов в батче
        h_dict = model(batch.x_dict, batch.edge_index_dict)
        
        # 2. Вычисляем предсказания для конкретного типа связи
        # Используем edge_label_index, который подготовил лоадер
        preds = classifier(
            h_dict['AA'], 
            h_dict['SmallMolecule'], 
            batch['AA', 'interacts_with', 'SmallMolecule'].edge_label_index
        )
        
        # 3. Считаем Loss, сравнивая с edge_label (1 или 0)
        target = batch['AA', 'interacts_with', 'SmallMolecule'].edge_label
        loss = criterion(preds, target)
        
        loss.backward()
        optimizer.step()